In [1]:
# Cell 1: cài package cần thiết (chạy 1 lần)
%pip install -q requests pypdf pymupdf easyocr pillow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 2: cấu hình backend + tạo session RAG
import os
import requests

BACKEND_URL = os.getenv("DOCMIND_BACKEND_URL", "http://localhost:8001").rstrip("/")
PDF_PATH = "/home/jkl/Code/VLLM-PD/documents/test1.pdf"  # đổi file nếu cần

# Health check
health = requests.get(f"{BACKEND_URL}/health", timeout=10).json()
print("Health:", health)

# Tạo session
session_resp = requests.post(f"{BACKEND_URL}/sessions", timeout=20)
session_resp.raise_for_status()
SESSION_ID = session_resp.json()["session_id"]
print("SESSION_ID:", SESSION_ID)

Loaded chars: 1745


In [ ]:
# Cell 3: upload PDF -> backend xử lý chunk + OCR + embedding + lưu FAISS
from pathlib import Path

pdf = Path(PDF_PATH)
assert pdf.exists(), f"Không tìm thấy file: {pdf}"

with pdf.open("rb") as f:
    files = {"file": (pdf.name, f, "application/pdf")}
    up = requests.post(
        f"{BACKEND_URL}/sessions/{SESSION_ID}/upload",
        files=files,
        timeout=600,
    )

up.raise_for_status()
upload_result = up.json()
print("Upload result:", upload_result)

Dưới đây là tóm tắt tài liệu "Multilingual Voice Interaction System for Humanoid Robot (Unitree G1)" thành 5 ý chính:

**1. Mục tiêu và yêu cầu của hệ thống**
Hệ thống Multilingual Voice Interaction System được thiết kế để hỗ trợ giao tiếp bằng giọng nói tự nhiên cho robot humanoid Unitree G1. Hệ thống cần hỗ trợ tiếng Việt và tiếng Nhật, hoạt động trong môi trường đa người nói và được triển khai trên thiết bị Jetson Orin.

**2. Kiến trúc và giải pháp**
Hệ thống được thiết kế với kiến trúc pipeline thời gian thực, bao gồm các thành phần sau: 
- Mic → Wakeword → VAD + ASR (nhận diện giọng nói và phân tích ý định)
- Intent Cache → LLM + RAG (giải quyết ý định và tạo ra câu trả lời)
- TTS → Audio Normalization → Speaker (trình bày câu trả lời)
- Client (Jetson) + Server (Cloud/Dev) (cơ sở dữ liệu và xử lý)

**3. Thách thức và hạn chế**
Hệ thống gặp phải các thách thức và hạn chế sau:
- Mismatch sample rate (khả năng không đồng bộ về tốc độ lấy mẫu)
- Buffer timing misalignment (khả năng k

In [ ]:
# Cell 4: query RAG và xem nguồn trích dẫn
query_payload = {
    "session_id": SESSION_ID,
    "question": "Tóm tắt tài liệu thành 5 ý chính và nêu các chi tiết quan trọng trong hình ảnh/biểu đồ (nếu có).",
    "stream": False,
}

resp = requests.post(f"{BACKEND_URL}/query", json=query_payload, timeout=300)
resp.raise_for_status()
data = resp.json()

print("\n===== ANSWER =====\n")
print(data["answer"])

print("\n===== SOURCES =====")
for i, s in enumerate(data.get("sources", []), 1):
    print(f"{i}. file={s.get('file')} | page={s.get('page')} | score={s.get('score')}")

Màu chủ đạo của tài liệu này là màu xám.


In [ ]:
# Cell 5: kiểm chứng local OCR ảnh trích từ PDF (để verify pipeline ảnh)
import io
import fitz  # pymupdf
import numpy as np
import easyocr
from PIL import Image

doc = fitz.open(PDF_PATH)
reader = easyocr.Reader(["vi", "en"], gpu=False)

total_images = 0
ocr_samples = []

for page_idx in range(len(doc)):
    page = doc[page_idx]
    for img in page.get_images(full=True):
        total_images += 1
        xref = img[0]
        base_img = doc.extract_image(xref)
        img_bytes = base_img["image"]

        pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        arr = np.array(pil_img)

        ocr_result = reader.readtext(arr, detail=0)
        text_joined = " ".join(ocr_result).strip()
        if text_joined:
            ocr_samples.append({
                "page": page_idx + 1,
                "text": text_joined[:300],
            })

print(f"Tổng số ảnh trích từ PDF: {total_images}")
print(f"Số ảnh có OCR text: {len(ocr_samples)}")

for i, item in enumerate(ocr_samples[:5], 1):
    print(f"[{i}] Page {item['page']}: {item['text']}")